# 📊 Notebook 1: Exploratory Data Analysis (EDA)
## AI-Driven Credit Risk Evaluation
**Author:** Bandaru Yashwanth | B.Sc. Actuarial Science, Amity University Noida

---
### 🎯 Objective
Understand the structure, distributions, and patterns in 10,000+ customer financial records before building a predictive model.

### 📋 EDA Checklist
1. Load & inspect data
2. Check shape, types, missing values
3. Descriptive statistics
4. Target variable distribution
5. Univariate analysis (each feature)
6. Bivariate analysis (features vs default)
7. Correlation analysis
8. Key insights summary

In [ ]:
# ── Import Libraries ───────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style settings
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
sns.set_palette('husl')

print('✅ Libraries loaded successfully')

## 1️⃣ Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/credit_risk_raw.csv')

print('=== DATASET OVERVIEW ===')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nColumns: {list(df.columns)}')
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Data types and null counts
print('=== DATA TYPES & MISSING VALUES ===')
info = pd.DataFrame({
    'dtype': df.dtypes,
    'null_count': df.isnull().sum(),
    'null_%': (df.isnull().sum() / len(df) * 100).round(2),
    'unique_values': df.nunique()
})
print(info)

In [ ]:
# Statistical summary
print('=== DESCRIPTIVE STATISTICS (Numerical Features) ===')
df.describe().round(2)

## 2️⃣ Target Variable — Default Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
colors = ['#2ECC71', '#E74C3C']
counts = df['default'].value_counts()
axes[0].bar(['No Default (0)', 'Default (1)'], counts.values, color=colors, edgecolor='black', linewidth=0.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')
axes[0].set_title('Target Variable Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(counts.values, labels=['No Default', 'Default'],
            autopct='%1.1f%%', colors=colors, startangle=90,
            explode=(0, 0.05), shadow=True)
axes[1].set_title('Default Rate', fontsize=14, fontweight='bold')

plt.suptitle('Credit Default Distribution (10,000 customers)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

default_rate = df['default'].mean() * 100
print(f'\n📊 Default Rate: {default_rate:.1f}%')
print(f'✅ Non-Default: {(100-default_rate):.1f}%')
print('⚠️  Class imbalance present — will need to handle in modeling step')

## 3️⃣ Univariate Analysis — Numerical Features

In [ ]:
num_cols = ['age', 'annual_income', 'loan_amount', 'credit_score',
            'employment_years', 'existing_debt', 'num_credit_lines', 'num_dependents']

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=40, color='#3498DB', edgecolor='white', linewidth=0.3, alpha=0.85)
    axes[i].axvline(df[col].median(), color='#E74C3C', linestyle='--', linewidth=2, label=f'Median: {df[col].median():.0f}')
    axes[i].axvline(df[col].mean(), color='#F39C12', linestyle='--', linewidth=2, label=f'Mean: {df[col].mean():.0f}')
    axes[i].set_title(f'{col.replace("_"," ").title()}', fontweight='bold')
    axes[i].legend(fontsize=9)
    axes[i].set_ylabel('Frequency')

plt.suptitle('Distribution of Numerical Features', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/numerical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Observations:')
print('  • Annual income is right-skewed → some high earners pulling mean up')
print('  • Credit score roughly normally distributed around 650')
print('  • Employment years is right-skewed → many newer employees in dataset')

## 4️⃣ Univariate Analysis — Categorical Features

In [ ]:
cat_cols = ['employment_type', 'education', 'property_ownership', 'loan_purpose', 'loan_term_months']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

palette = sns.color_palette('Set2')
for i, col in enumerate(cat_cols):
    vc = df[col].value_counts()
    bars = axes[i].bar(vc.index.astype(str), vc.values, color=palette[:len(vc)], edgecolor='black', linewidth=0.4)
    for bar, val in zip(bars, vc.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                     f'{val:,}', ha='center', fontsize=9, fontweight='bold')
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].tick_params(axis='x', rotation=20)

axes[5].set_visible(False)
plt.suptitle('Distribution of Categorical Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5️⃣ Bivariate Analysis — Features vs Default Rate

In [ ]:
# Default rate by categorical features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(['employment_type', 'education', 'property_ownership', 'loan_purpose']):
    dr = df.groupby(col)['default'].mean().sort_values(ascending=False) * 100
    colors_bar = ['#E74C3C' if v > df['default'].mean()*100 else '#2ECC71' for v in dr.values]
    bars = axes[i].bar(dr.index, dr.values, color=colors_bar, edgecolor='black', linewidth=0.4)
    axes[i].axhline(df['default'].mean()*100, color='navy', linestyle='--', linewidth=1.5,
                    label=f'Overall avg: {df["default"].mean()*100:.1f}%')
    for bar, val in zip(bars, dr.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                     f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')
    axes[i].set_title(f'Default Rate by {col.replace("_"," ").title()}', fontweight='bold')
    axes[i].set_ylabel('Default Rate (%)')
    axes[i].legend()
    axes[i].tick_params(axis='x', rotation=15)

plt.suptitle('Default Rate by Categorical Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/default_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Boxplots: key numerical features vs default
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

key_cols = ['credit_score', 'annual_income', 'loan_amount',
            'employment_years', 'existing_debt', 'num_dependents']

for i, col in enumerate(key_cols):
    df.boxplot(column=col, by='default', ax=axes[i],
               boxprops=dict(color='#2C3E50'),
               medianprops=dict(color='#E74C3C', linewidth=2))
    axes[i].set_title(f'{col.replace("_"," ").title()} vs Default', fontweight='bold')
    axes[i].set_xlabel('Default (0=No, 1=Yes)')
    axes[i].set_xticklabels(['No Default', 'Default'])

plt.suptitle('Key Features vs Default Status', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/boxplots_vs_default.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n🔍 Key Observations from Boxplots:')
for col in key_cols:
    m0 = df[df['default']==0][col].median()
    m1 = df[df['default']==1][col].median()
    diff = ((m1-m0)/m0*100) if m0!=0 else 0
    direction = '↑' if diff > 0 else '↓'
    print(f'  • {col}: No Default median={m0:.0f}, Default median={m1:.0f} ({direction}{abs(diff):.0f}%)')

## 6️⃣ Correlation Analysis

In [ ]:
# Correlation heatmap
corr_cols = ['age', 'annual_income', 'loan_amount', 'credit_score',
             'employment_years', 'existing_debt', 'num_credit_lines',
             'num_dependents', 'loan_term_months', 'default']

corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn_r', center=0, vmin=-1, vmax=1,
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 9})
plt.title('Correlation Matrix — Credit Risk Features', fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../data/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlation with target
print('\n📊 Correlation with Default (sorted):')
target_corr = corr_matrix['default'].drop('default').sort_values(key=abs, ascending=False)
for feat, corr in target_corr.items():
    bar = '█' * int(abs(corr) * 30)
    sign = '+' if corr > 0 else '-'
    print(f'  {feat:<22} {sign}{abs(corr):.3f}  {bar}')

## 7️⃣ Outlier Detection

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col in zip(axes, ['annual_income', 'loan_amount', 'existing_debt']):
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]

    ax.boxplot(df[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='#3498DB', alpha=0.7),
               medianprops=dict(color='red', linewidth=2),
               flierprops=dict(marker='o', color='#E74C3C', alpha=0.5))
    ax.set_title(f'{col.replace("_"," ").title()}\n({len(outliers)} outliers = {len(outliers)/len(df)*100:.1f}%)', fontweight='bold')
    ax.set_xticks([])

plt.suptitle('Outlier Detection (IQR Method)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/outlier_detection.png', dpi=150, bbox_inches='tight')
plt.show()

## 8️⃣ EDA Summary — Key Insights

In [ ]:
print('=' * 60)
print('  📋 EDA SUMMARY — KEY INSIGHTS')
print('=' * 60)
print()
print('📊 Dataset:')
print(f'  • {len(df):,} customer records, {df.shape[1]} features')
print(f'  • Default rate: {df["default"].mean()*100:.1f}% (moderate imbalance)')
print(f'  • Missing values in: credit_score, employment_years, num_credit_lines')
print()
print('🔍 Risk Drivers Found:')
print('  1. Credit Score   — lower scores strongly associated with default')
print('  2. Existing Debt  — higher debt-to-income = higher default risk')
print('  3. Loan Amount    — larger loans default more')
print('  4. Employment     — Freelancers and Self-Employed default more')
print('  5. Property       — Renters default more than property owners')
print()
print('⚠️  Preprocessing Needed:')
print('  • Impute missing values (median for numeric)')
print('  • Encode categorical variables (One-Hot / Label)')
print('  • Feature engineering (debt-to-income ratio, etc.)')
print('  • Handle class imbalance (class_weight or SMOTE)')
print()
print('✅ Ready to proceed to Notebook 2: Feature Engineering!')